In [1]:
import torch
import torch.nn.functional as F

In [2]:
words = open('names.txt', 'r').read().splitlines()
aphabets = sorted(list(set("".join(words))))
stoi = {s:i+1 for i,s in enumerate(aphabets)} # a starting from 1
stoi['.'] = 0
itos = {v:k for k,v in stoi.items()}

In [51]:
# build the dataset
block_size = 3
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)

In [4]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

### Implementing embedding lookup table

In [5]:
C = torch.randn((27, 2))
C[5]
# Approach 1 : One hot way as we did earlier (5, 27) x (27, 2) -> (5, 2) | 5 words 2 dim vector to represent word
# Approach 2 : Direct use C as a look up table as 

tensor([0.7994, 0.6466])

In [9]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C # C is just accessing rows!

tensor([0.7994, 0.6466])

In [12]:
C[torch.tensor([5, 6, 7, 7, 7, 7, 7, 7])]

tensor([[ 0.7994,  0.6466],
        [-0.2613,  0.0933],
        [ 0.7818, -0.1936],
        [ 0.7818, -0.1936],
        [ 0.7818, -0.1936],
        [ 0.7818, -0.1936],
        [ 0.7818, -0.1936],
        [ 0.7818, -0.1936]])

In [14]:
C[X].shape

torch.Size([32, 3, 2])

In [15]:
embedding = C[X]
embedding.shape

torch.Size([32, 3, 2])

In [21]:
# torch.cat([embedding[:, 0, :], embedding[:, 0, :], embedding[:, 0, :]], 1).shape

torch.cat(torch.unbind(embedding, 1), 1).shape

torch.Size([32, 6])

In [22]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)
h = torch.tanh(embedding.view(-1, 6) @ W1 + b1)

# 32, 100
#  1, 100 #fake dimension 1

In [23]:
h.shape

torch.Size([32, 100])

In [24]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)
logits = h @ W2 + b2
logits.shape

torch.Size([32, 27])

In [25]:
counts = logits.exp()
prob = counts / counts.sum(1, keepdim=True)

In [26]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [27]:
torch.arange(32) # number of rows/ training examples

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [31]:
loss = -prob[torch.arange(32), Y].log().mean() # probabilties against ground truth output
loss

tensor(13.8257)

### Condensed form

In [67]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100))
b1 = torch.randn(100)
W2 = torch.randn((100, 27))
b2 = torch.randn(27)
parameters = [C, W1, b1, W2, b2]

In [68]:
sum(p.nelement() for p in parameters)
for p in parameters:
    p.requires_grad = True

In [69]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10 ** lre

In [70]:
lr = []
losses = []
for i in range(100):
    ix = torch.randint(0, X.shape[0], (32, ))
        # Forward Pass
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ix])
    # Backward Pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -lrs[i] * p.grad
    lr.append(lrs[i])
    losses.append(loss.item())
loss

tensor(13.6315, grad_fn=<NllLossBackward0>)

In [71]:
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
loss

tensor(15.9371, grad_fn=<NllLossBackward0>)